# 从最小训练循环理解 PyTorch 训练过程

这份 Notebook 的主线只有一句话：**取一批数据 → 前向计算 loss → 反向计算梯度 → 用梯度更新参数**。验证、日志、checkpoint、梯度累积、裁剪和混合精度，都是围绕这四步增加的控制机制。

## 学习目标

完成后，你应该能够：

- 独立写出最小 PyTorch 训练循环，并解释每一行的作用；
- 说明 `zero_grad()`、`backward()` 和 `step()` 为什么必须按顺序出现；
- 区分 batch、iteration 和 epoch；
- 正确组织训练与验证，并理解 `train()`、`eval()`、`no_grad()`；
- 知道日志、checkpoint、梯度累积、梯度裁剪和 AMP 应插在哪里；
- 根据 loss、梯度和参数是否变化定位常见错误。

## 前置知识

默认你了解 Python 函数、张量和最基本的导数概念。不要求提前掌握 autograd。所有实验使用很小的合成数据，不下载数据集，也不需要 GPU。

In [ ]:
from pathlib import Path
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

## 1. 核心心智模型

设模型为 $f_\theta$，参数为 $\theta$。一个 batch 的训练可以写成：

$$\hat y=f_\theta(x), \qquad L=\frac{1}{B}\sum_{i=1}^{B}\ell(\hat y_i,y_i)$$

反向传播得到 $g=\nabla_\theta L$，优化器以学习率 $\eta$ 更新参数。最简单的 SGD 是：

$$\theta \leftarrow \theta-\eta g$$

把它翻译成 PyTorch，就是下面四个动作：

1. `pred = model(x)`：用当前参数做前向传播并建立计算图；
2. `loss = loss_fn(pred, y)`：把预测误差压缩为一个标量目标；
3. `loss.backward()`：沿计算图用链式法则计算每个参数的梯度；
4. `optimizer.step()`：优化器读取梯度并原地更新参数。

`optimizer.zero_grad()` 不属于数学上的优化步骤，但在 PyTorch 中不可忽略，因为参数的 `.grad` 默认会**累加**。它通常放在下一次 `backward()` 之前。

## 2. 准备一个看得懂的数据问题

我们学习线性关系 $y=3x+2+\epsilon$。每个样本的 `x` 和 `y` 形状都是 `[1]`；`DataLoader` 拼成 batch 后形状为 `[batch_size, 1]`。数据被拆为训练集和验证集，验证集只评价，不参与更新参数。

In [ ]:
generator = torch.Generator().manual_seed(SEED)
x = torch.linspace(-2, 2, 160).unsqueeze(1)
noise = 0.15 * torch.randn(x.shape, generator=generator)
y = 3 * x + 2 + noise

train_dataset = TensorDataset(x[:128], y[:128])
val_dataset = TensorDataset(x[128:], y[128:])
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, generator=generator)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

batch_x, batch_y = next(iter(train_loader))
print("number of train samples:", len(train_dataset))
print("iterations per epoch:", len(train_loader))
print("batch shapes:", batch_x.shape, batch_y.shape)

### batch、iteration、epoch

- **batch**：一次送入模型的一组样本；这里通常是 16 个。
- **iteration/step**：处理一个 batch 并更新一次参数。
- **epoch**：训练集中的样本大致都被使用一次；这里一个 epoch 有 `len(train_loader) = 8` 次 iteration。

`shuffle=True` 会在每个 epoch 重排训练样本；验证阶段通常不需要 shuffle。

## 3. 一次参数更新：把每个状态都摊开看

下面只训练一个 batch，并观察三种状态：loss、梯度、参数。注意 `optimizer.step()` 不会自动清空梯度。

In [ ]:
torch.manual_seed(SEED)
model = nn.Linear(1, 1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

batch_x, batch_y = batch_x.to(device), batch_y.to(device)
weight_before = model.weight.detach().clone()

optimizer.zero_grad()                 # 1. 清除上一次留下的梯度
pred = model(batch_x)                 # 2. forward：预测并建立计算图
loss = loss_fn(pred, batch_y)         # 3. 得到标量优化目标
loss.backward()                       # 4. backward：把梯度写入 parameter.grad
gradient = model.weight.grad.detach().clone()
optimizer.step()                      # 5. 更新 parameter 本身

print("loss:", loss.item())
print("weight before:", weight_before.item())
print("weight gradient:", gradient.item())
print("weight after:", model.weight.item())
print("weight changed:", not torch.equal(weight_before, model.weight.detach()))

### 为什么顺序重要？

- 在 `backward()` 前没有 forward/loss，就没有可求导的计算图。
- 在 `backward()` 前调用 `step()`，优化器读不到本轮梯度。
- 忘记 `zero_grad()`，本轮梯度会加到旧梯度上；普通训练会因此改变更新尺度。
- 把 `zero_grad()` 放在 `backward()` 与 `step()` 之间，会把刚算出的梯度删除，参数无法更新。

常见写法也可以在 `step()` 后清梯度；关键约束是：**每个普通 iteration 的 `backward()` 前，旧梯度已经被清除**。`set_to_none=True` 通常更省内存，并能让“没有梯度”和“梯度为零”更容易区分。

## 4. 最小训练循环

现在只增加两层循环：外层遍历 epoch，内层遍历 batch。为了正确计算 epoch 平均 loss，我们用 `loss × batch 样本数` 累加，最后除以样本总数；这样最后一个不足整 batch 时也不会产生偏差。

In [ ]:
torch.manual_seed(SEED)
model = nn.Linear(1, 1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(20):
    model.train()
    loss_sum = 0.0
    sample_count = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad(set_to_none=True)
        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)
        loss.backward()
        optimizer.step()

        batch_size = batch_x.size(0)
        loss_sum += loss.item() * batch_size
        sample_count += batch_size

    train_loss = loss_sum / sample_count
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"epoch={epoch + 1:02d} train_loss={train_loss:.4f}")

print(f"learned: y = {model.weight.item():.3f}x + {model.bias.item():.3f}")

到这里，最小训练循环已经完整。以后遇到复杂 Trainer，可以先寻找这五行：`zero_grad → model → loss → backward → step`。数据搬运、日志、调度器和分布式通信最终都围绕它们排列。

## 5. 验证循环与三组容易混淆的开关

`model.train()` 和 `model.eval()` 改变部分层的**行为模式**：典型的是 Dropout 与 BatchNorm。它们不会控制 autograd。`torch.no_grad()`（或更严格的 `torch.inference_mode()`）控制是否记录反向传播需要的信息，但不会自动切换层的行为。

因此验证通常同时需要：

```python
model.eval()
with torch.no_grad():
    ...
```

验证阶段没有 `backward()` 和 `optimizer.step()`，因为验证数据不能影响模型参数。

In [ ]:
def evaluate(model, data_loader, loss_fn, device):
    model.eval()
    loss_sum = 0.0
    sample_count = 0

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            pred = model(batch_x)
            loss = loss_fn(pred, batch_y)
            loss_sum += loss.item() * batch_x.size(0)
            sample_count += batch_x.size(0)

    return loss_sum / sample_count

val_loss = evaluate(model, val_loader, loss_fn, device)
print(f"validation loss: {val_loss:.4f}")
print("model.training after evaluate:", model.training)

函数结束后模型仍处于 eval 模式。因此下一轮训练开头应再次调用 `model.train()`。一个常见工程写法是分别定义 `train_one_epoch(...)` 和 `evaluate(...)`，由外层 epoch 循环调度。

## 6. 一个结构清晰、仍然很小的版本

这个版本没有引入 Trainer 类，只把职责拆成函数。它已经包含训练、验证和日志，是多数项目进一步扩展的合适起点。

In [ ]:
def train_one_epoch(model, data_loader, loss_fn, optimizer, device):
    model.train()
    loss_sum = 0.0
    sample_count = 0

    for batch_x, batch_y in data_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * batch_x.size(0)
        sample_count += batch_x.size(0)

    return loss_sum / sample_count


torch.manual_seed(SEED)
clean_model = nn.Linear(1, 1).to(device)
clean_optimizer = torch.optim.SGD(clean_model.parameters(), lr=0.05)
history = []

for epoch in range(10):
    train_loss = train_one_epoch(clean_model, train_loader, loss_fn, clean_optimizer, device)
    val_loss = evaluate(clean_model, val_loader, loss_fn, device)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})

for row in history:
    print(f"epoch={row['epoch']:02d} train={row['train_loss']:.4f} val={row['val_loss']:.4f}")

## 7. 日志、checkpoint 与恢复训练

日志回答“训练发生了什么”；checkpoint 回答“如何从某个状态继续”。要**继续训练**，只保存模型参数通常不够，还应保存优化器状态和已完成的 epoch。Adam 等优化器内部维护动量；丢失这些状态会使恢复后的更新轨迹发生突变。

下面把 checkpoint 写到系统临时目录，避免污染仓库。真实项目还可能保存 scheduler、AMP scaler、随机数状态和配置。不要反序列化不可信来源的 checkpoint。

In [ ]:
import tempfile

checkpoint_path = Path(tempfile.gettempdir()) / "training-loop-demo.pt"
checkpoint = {
    "epoch": history[-1]["epoch"],
    "model_state": clean_model.state_dict(),
    "optimizer_state": clean_optimizer.state_dict(),
    "history": history,
}
torch.save(checkpoint, checkpoint_path)

restored_model = nn.Linear(1, 1).to(device)
restored_optimizer = torch.optim.SGD(restored_model.parameters(), lr=0.05)
loaded = torch.load(checkpoint_path, map_location=device, weights_only=True)
restored_model.load_state_dict(loaded["model_state"])
restored_optimizer.load_state_dict(loaded["optimizer_state"])

same_parameters = all(
    torch.equal(a, b)
    for a, b in zip(clean_model.parameters(), restored_model.parameters())
)
print("saved after epoch:", loaded["epoch"])
print("parameters restored exactly:", same_parameters)

## 8. 三个高级扩展放在哪里？

这些机制值得认识，但先不要让它们遮住最小循环。

### 8.1 梯度累积

显存只能容纳小 batch 时，可以连续处理 $K$ 个 micro-batch，再更新一次参数。由于 loss 默认是 batch 均值，每次应使用 `loss / K`，使最终梯度尺度近似一个大 batch。此时不能每个 micro-batch 都 `zero_grad()`。

```python
optimizer.zero_grad(set_to_none=True)
for step, (x, y) in enumerate(loader):
    loss = loss_fn(model(x), y) / accumulation_steps
    loss.backward()
    if (step + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
```

真实实现还要处理最后不足 $K$ 个 micro-batch 的情况。BatchNorm 等依赖 batch 统计量的层，也不会因此完全等价于真正的大 batch。

### 8.2 梯度裁剪

裁剪必须放在 `backward()` 之后、`step()` 之前，因为只有这时梯度已经存在且尚未被使用：

```python
loss.backward()
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
```

它常用于抑制 exploding gradients，但不应代替对错误学习率、异常数据或数值不稳定的排查。

### 8.3 自动混合精度（AMP）

CUDA 上的典型结构是：在 autocast 中做 forward/loss，用 GradScaler 缩放 loss 后 backward；若要裁剪，应先 `unscale_`。具体 API 会随 PyTorch 版本和设备能力变化：

```python
optimizer.zero_grad(set_to_none=True)
with torch.autocast(device_type="cuda", dtype=torch.float16):
    loss = loss_fn(model(x), y)
scaler.scale(loss).backward()
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()
```

AMP 的目标是提升吞吐并降低显存占用，不保证所有模型都更快，也不意味着参数和运算全部变成低精度。

## 9. 用可观察信号排查训练循环

| 现象 | 优先检查 | 常见原因 |
| --- | --- | --- |
| loss 完全不变 | 参数更新前后是否不同 | 忘记 `backward()`/`step()`，参数被冻结，学习率为 0 |
| `.grad is None` | 参数是否参与 forward | 计算图断开、误用 `detach()`、该分支未使用参数 |
| 梯度每步越来越大 | 清梯度的位置 | 忘记 `zero_grad()`，或本应累积但未缩放 loss |
| loss 为 NaN/Inf | 输入、loss、梯度逐层检查 | 学习率过大、非法数学运算、数值溢出 |
| 训练好、验证差异异常 | 模式开关 | 验证时忘记 `eval()`，数据预处理不一致 |
| GPU 很慢 | 数据和同步点 | 频繁 `.item()`、数据加载慢、batch 太小 |
| 恢复训练后突然波动 | checkpoint 内容 | 没恢复 optimizer/scheduler/scaler 状态 |

推荐按数据、forward、loss、gradient、parameter update 的顺序定位，而不是先重写整个 Trainer。

In [ ]:
# 一个很小的“训练循环体检”：loss 有限、梯度存在、参数确实改变。
debug_model = nn.Linear(1, 1).to(device)
debug_optimizer = torch.optim.SGD(debug_model.parameters(), lr=0.01)
debug_x, debug_y = next(iter(train_loader))
debug_x, debug_y = debug_x.to(device), debug_y.to(device)

debug_optimizer.zero_grad(set_to_none=True)
debug_loss = loss_fn(debug_model(debug_x), debug_y)
assert torch.isfinite(debug_loss), "loss 不是有限值"
debug_loss.backward()
assert all(p.grad is not None for p in debug_model.parameters()), "存在未获得梯度的参数"
before = [p.detach().clone() for p in debug_model.parameters()]
debug_optimizer.step()
assert any(not torch.equal(a, b) for a, b in zip(before, debug_model.parameters())), "参数没有更新"
print("basic training-loop checks passed")

## 10. 自测与练习

先不运行新代码，尝试回答：

1. 为什么 `optimizer.step()` 不放在 `torch.no_grad()` 中也能安全更新参数？
2. 如果忘记 `model.eval()`，但使用了 `torch.no_grad()`，带 Dropout 的模型会怎样？
3. 为什么记录 epoch loss 时不能简单平均每个 batch 的 `loss.item()`？
4. 梯度累积 4 次时，为什么通常要把每个 micro-batch 的 loss 除以 4？
5. checkpoint 只保存 `model.state_dict()`，适合推理还是无缝恢复训练？

动手练习：

- **基础**：删掉最小循环中的一行，先预测会出现什么症状，再运行验证。
- **基础**：把模型改成 `nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))`，观察 loss。
- **进阶**：为 `train_one_epoch` 增加 `max_grad_norm` 可选参数，并记录裁剪前梯度范数。
- **进阶**：实现能正确处理最后残余 micro-batch 的梯度累积。
- **工程**：在 checkpoint 中加入随机数状态，验证恢复后下一批 shuffle 顺序一致。

## 一页总结

```text
for epoch:
    model.train()
    for batch:
        move data to device
        zero_grad()
        prediction = model(input)
        loss = loss_fn(prediction, target)
        loss.backward()
        [gradient clipping]
        optimizer.step()
        record metrics

    model.eval()
    with no_grad(): validate
    save checkpoint when appropriate
```

最重要的不是背下一套框架，而是始终追踪三类状态：**数据是什么、梯度在哪里、参数何时改变**。

## 相关知识与下一步

当前仓库还没有直接相关的独立知识条目。自然的下一步是 autograd 与计算图、优化器与学习率、数据加载性能，以及混合精度训练；当其中某一项需要系统展开时，应建立新的小主题，而不是无限扩张本 Notebook。